In [1]:
import pandas as pd
import numpy as np
# pandas: reading CSV/Excel, handling missing values, filtering, grouping
# numpy: numerical operations, NaN handling, vectorized logic
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# dataframe
df=pd.read_csv("uberDataset.csv")

In [3]:
df.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  object 
 1   Time                               150000 non-null  object 
 2   Booking ID                         150000 non-null  object 
 3   Booking Status                     150000 non-null  object 
 4   Customer ID                        150000 non-null  object 
 5   Vehicle Type                       150000 non-null  object 
 6   Pickup Location                    150000 non-null  object 
 7   Drop Location                      150000 non-null  object 
 8   Avg VTAT                           139500 non-null  float64
 9   Avg CTAT                           102000 non-null  float64
 10  Cancelled Rides by Customer        10500 non-null   float64
 11  Reason for cancelling by Customer  1050

In [5]:
rows, columns = df.shape

print("Rows:", rows)
print("Columns:", columns)

Rows: 150000
Columns: 21


In [11]:
# Always check:
df.isnull().sum().sort_values(ascending=False)

incomplete_rides_reason              141000
incomplete_rides                     141000
reason_for_cancelling_by_customer    139500
cancelled_rides_by_customer          139500
cancelled_rides_by_driver            123000
driver_cancellation_reason           123000
driver_ratings                        57000
customer_rating                       57000
avg_ctat                              48000
ride_distance                         48000
booking_value                         48000
payment_method                        48000
avg_vtat                              10500
customer_id                               0
booking_status                            0
booking_id                                0
time                                      0
date                                      0
pickup_location                           0
drop_location                             0
vehicle_type                              0
dtype: int64

In [12]:
# Standardize Column Names (VERY IMPORTANT)
df.columns=(
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ","_")
)
df.head()


,date,time,booking_id,booking_status,customer_id,vehicle_type,pickup_location,drop_location,avg_vtat,avg_ctat,...,reason_for_cancelling_by_customer,cancelled_rides_by_driver,driver_cancellation_reason,incomplete_rides,incomplete_rides_reason,booking_value,ride_distance,driver_ratings,customer_rating,payment_method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [ ]:
# 3️⃣ Fix Date & Time Columns
df['date']=pd.to_datetime(df['date'],error='coerce')
df['time']=pd.to_datetime(df['time'],format='%H:%M:%S',errors='coerce').dt.time
df['month']=df['date'].dt.month
df['day']=df['date'].dt.day_name()

In [18]:
# 4️⃣ Handle Null (Missing) Values
no_driver_df = df[df['booking_status'] == 'No Driver Found']
completed_df = df[df['booking_status'] == 'Completed']
completed_df = completed_df.dropna(
    subset=['booking_value', 'ride_distance']
)
df.groupby("booking_status")["ride_distance"].mean()

booking_status
Cancelled by Customer          NaN
Cancelled by Driver            NaN
Completed                26.000493
Incomplete               10.547706
No Driver Found                NaN
Name: ride_distance, dtype: float64